In [ ]:
from option_analyzer import *
from indicators import compute_emas
self = OptionAnalyzer('quotes', 'chain')

### Run this once every day to load close prices in the past 60 days

In [ ]:
df_close = pd.read_csv('output/yf_close.csv', parse_dates=['Date']).set_index('Date').tail(60)
df_close.columns.name = 'symbol'
latest_bollinger_file = max(glob('output/bollinger*.csv'))
df_boll = pd.read_csv(latest_bollinger_file).set_index('symbol')
df_boll = df_boll.loc[:, ['LB20', 'UB20', 'MA20', 'LB30', 'UB30', 'MA30']]
today = pd.Timestamp.now().normalize()
print('df_close data age:', today - df_close.index[-1])
print('latest Bollinger file:', latest_bollinger_file)

### Either run these two cells

### Or run this cell to read from data directory

In [ ]:
os.system('sync > /dev/null 2>&1')
option_type = 'put'
servers = sorted(set([f.split('~')[1] for f in glob(os.path.expanduser(f'~/lab/data/{option_type}~*~*.csv'))]))
latest_option_files = [sorted(glob(os.path.expanduser(f'~/lab/data/{option_type}~{svr}~*.csv')))[-1] for svr in servers]
print('\n'.join(['%40s' % _ for _ in map(os.path.basename, latest_option_files)]))
chain_file_mtimes = dict([(os.path.basename(_f), os.path.getmtime(_f)) for _f in glob(os.path.expanduser('~/lab/chain/*'))])
latest_symbol = sorted(chain_file_mtimes, key=chain_file_mtimes.get)[-1]
print('Last symbol:', latest_symbol, datetime.fromtimestamp(chain_file_mtimes[latest_symbol]).strftime('%F %T'))
dfp = pd.concat([pd.read_csv(_f) for _f in latest_option_files])
df_today = dfp.loc[:, ['symbol', 'lastPrice']].drop_duplicates().rename(columns={'lastPrice': today}).set_index('symbol')
if df_today.columns[0] in df_close.T.columns:
    df_price = df_close
else:
    df_price = df_close.T.join(df_today, how='right').T
df_ema = compute_emas(df_price, [21, 50])

### Put options with no earning date on or before expiration date
- Sell puts to maximize hdteProfit.
- hdte_resid should be < 0.5, maybe even 0.4.
- It's okay to have high spread because half of the spreads have been deducted from hdte profit.

In [ ]:
dfp['hdtePriceOvStrike'] = 100*dfp.lastPrice*(1 - dfp.ImpVola * np.sqrt(dfp.dte/730))/dfp.strike - 100
_filter = (dfp.hdtePriceOvStrike >= 3) & (dfp.hdteProfit >= 24) & (dfp.mid >= 0.5)
#hdte_resid_ub = 0.5
spread_ub = 25
#premium_lb = 1
#delta_lb = -0.25
_filter = _filter & (dfp.pctSpread <= spread_ub)
#_filter = _filter & (dfp.hdte_resid<=hdte_resid_ub)
#_filter = _filter & (dfp.mid >= premium_lb) & (dfp.Delta >= delta_lb)
_filter = _filter & (~dfp.symbol.str.contains(r'^(?:SNDK|MU|LITE)'))
#_filter = _filter & (dfp.E.isna() |(dfp.E > dfp.dte)) # Note: Fidelity's earning report dates are not reliable
_dfp = dfp[_filter].sort_values(by='hdteProfit', ascending=False)
_history_csv = pd.Timestamp.now().strftime('history/put~%F~%T.csv')
_dfp.to_csv(_history_csv, index=None)
print(_history_csv, os.path.getsize(_history_csv), 'bytes archived')
print('Options after the filters:', _dfp.shape[0], 'out of', dfp.shape[0])
px.scatter(_dfp.head(200), x='hdtePriceOvStrike', y='hdteProfit', color='symbol', height=550).show()

_dfb = df_boll.join(df_today, how='right')
_dfb['rank20'] = (_dfb[today].astype(float) - _dfb['MA20'].astype(float))/(_dfb['UB20'] - _dfb['LB20'])*200
_dfb['rank30'] = (_dfb[today].astype(float) - _dfb['MA30'].astype(float))/(_dfb['UB30'] - _dfb['LB30'])*200
#_dfb = _dfb.loc[list(_dfp.symbol)]
px.bar(_dfb.sort_index(), y=['rank20', 'rank30'], barmode='group').show()

df_last_ema = df_price.tail(1).T.join(df_ema.tail(1).T.unstack(level=1).droplevel(level=0, axis=1))
df_last_ema = df_last_ema.sort_index()
px.bar(df_last_ema.loc[:, reversed(df_last_ema.columns)], barmode='group').show()

_dfp.head(40)

In [ ]:
342417/370

In [ ]:
_dfp[(_dfp.symbol.str.contains(r'QQQ|SPY|IWM|IBIT|GLD|TLT|DIA|EWY'))].head(20)

In [ ]:
_symbol = 'SNDK'
px.line(pd.concat([df_ema[_symbol], df_price[_symbol]], axis=1))

### Put options for specific symbols

In [ ]:
_filter = dfp.symbol.str.contains('SNDK')#QQQ|SPY|GLD|IBIT|DIA')
_filter = _filter & (dfp.hdtePriceOvStrike >= 2) & (dfp.hdteProfit >= 24)
_dfp = dfp[_filter].sort_values(by='hdteProfit', ascending=False).head(100)
px.scatter(_dfp, x='Delta', y='hdteProfit', color='expDt', height=500).show()
px.scatter(_dfp[(_dfp.expDt=='2026-03-27')], x='Delta', y='hdteProfit', color='strike', height=500).show()
print(_dfp.shape)
_dfp.head(25)

### Put options: top 500 in-the-money

In [ ]:
_df = dfp[(dfp.moneyness <= 1)].sort_values(by='hdteProfit', ascending=False).head(100)
px.scatter(_df, x='hdte_resid', y='hdteProfit', color='symbol', height=600).show()
_df.head(60)